<a href="https://colab.research.google.com/github/AKSHAYA-1412/GEN-AI-TRAINING/blob/main/Asssignment_GEN_AI_1_Sentimental_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install torch torchvision torchtext datasets scikit-learn nltk

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 26.2 MB/s eta 0:00:00


In [ ]:
from datasets import load_dataset
#Hugging Face provides datasets mainly for NLP tasks.
dataset = load_dataset("imdb")
#Downloads dataset
#Stores in cache
#Next time → loads locally
train_data = dataset['train']
test_data = dataset['test']

print(train_data[0])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

{'text': 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far be

In [ ]:
import nltk
import re
from nltk.corpus import stopwords

nltk.download('stopwords')#common words that don’t add meaning
stop_words = set(stopwords.words('english'))

def preprocess(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)  # remove punctuation
    words = text.split()
    words = [w for w in words if w not in stop_words]
    return " ".join(words)

train_texts = [preprocess(x['text']) for x in train_data]
train_labels = [x['label'] for x in train_data]

test_texts = [preprocess(x['text']) for x in test_data]
test_labels = [x['label'] for x in test_data]

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [ ]:
#feature extraction method-1 term frequency-inverse document frequency
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=5000)#convert text to numbers #take most important 5000 words ignore rest

X_train = vectorizer.fit_transform(train_texts).toarray()#learn important words from training data
X_test = vectorizer.transform(test_texts).toarray()

In [ ]:
#feature extraction method-2 used to Counts words
from sklearn.feature_extraction.text import CountVectorizer

bow_vectorizer = CountVectorizer(max_features=5000)

X_train_bow = bow_vectorizer.fit_transform(train_texts).toarray()
X_test_bow = bow_vectorizer.transform(test_texts).toarray()


In [ ]:
from sklearn.linear_model import LogisticRegression

model_lr = LogisticRegression(max_iter=1000)  # create model and allow it to learn up to 1000 times
model_lr.fit(X_train, train_labels)# train the model using TF-IDF features and their correct labels
pred_lr = model_lr.predict(X_test)  # use trained model to predict sentiment for test data

model_bow = LogisticRegression(max_iter=1000)# create another model for Bag of Words features
model_bow.fit(X_train_bow, train_labels)  # train model using BoW features
pred_bow = model_bow.predict(X_test_bow)  # predict sentiment using BoW features

In [ ]:
from sklearn.naive_bayes import MultinomialNB

model_nb = MultinomialNB()
model_nb.fit(X_train, train_labels)

pred_nb = model_nb.predict(X_test)

model_nb_bow = MultinomialNB()
model_nb_bow.fit(X_train_bow, train_labels)

pred_nb_bow = model_nb_bow.predict(X_test_bow)


In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Accuracy
print("TF-IDF + Logistic Regression:", accuracy_score(test_labels, pred_lr))
print("TF-IDF + Naive Bayes:", accuracy_score(test_labels, pred_nb))

print("BoW + Logistic Regression:", accuracy_score(test_labels, pred_bow))
print("BoW + Naive Bayes:", accuracy_score(test_labels, pred_nb_bow))


# Classification Report (example: best model)
print("\nClassification Report (TF-IDF + Logistic Regression):")
print(classification_report(test_labels, pred_lr))


# Confusion Matrix (example: best model)
print("\nConfusion Matrix (TF-IDF + Logistic Regression):")
print(confusion_matrix(test_labels, pred_lr))

TF-IDF + Logistic Regression: 0.88024
TF-IDF + Naive Bayes: 0.84272
BoW + Logistic Regression: 0.84856
BoW + Naive Bayes: 0.83816

Classification Report (TF-IDF + Logistic Regression):
              precision    recall  f1-score   support

           0       0.88      0.88      0.88     12500
           1       0.88      0.88      0.88     12500

    accuracy                           0.88     25000
   macro avg       0.88      0.88      0.88     25000
weighted avg       0.88      0.88      0.88     25000


Confusion Matrix (TF-IDF + Logistic Regression):
[[10966  1534]
 [ 1460 11040]]
